In [ ]:
!pip install transformers datasets torch librosa scikit-learn

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio,  DatasetDict

In [ ]:
model_or = AutoModelForAudioClassification.from_pretrained("openai/whisper-base")
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/whisper-finetuned-TESS")

feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/whisper-finetuned-TESS")

model_or.eval()
model_ft.eval()

In [ ]:
#Φόρτωση Dataset και speaker eindependent split
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

In [ ]:
#Εξαγωγή των ετικετών των συναισθημάτων σε λίστα
labels = sorted(set(dataset["train"]["emotion"]))
print(labels)

#Δημιουργία λεξικών αντιστοίχισης ετικετών και αριθμών
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

#υπολογισμός αριθμού ετικετών
num_labels = len(labels)
print(f"Διαθέσιμες ετικέτες: {labels}")
print(f"Αριθμός ετικετών: {num_labels}")

In [ ]:
sample = test_dataset[0]

In [ ]:
input_values = feature_extractor(sample["audio"]["array"], sampling_rate=sample["audio"]["sampling_rate"], return_tensors="pt").input_features

input_values = input_values.to(model_ft.device)

In [ ]:
embeddings = []
labels = []

In [ ]:
for sample in test_dataset:

    input_values = feature_extractor(sample["audio"]["array"], sampling_rate=sample["audio"]["sampling_rate"], return_tensors="pt").input_features
    input_values = input_values.to(model_or.device)

    with torch.no_grad():
        outputs = model_or(
            input_features=input_values,
            output_hidden_states=True
        )

    hidden = outputs.hidden_states[-1]
    embedding = hidden.mean(dim=1)
    embeddings.append(embedding.squeeze().cpu().numpy())
    labels.append(label2id[sample["emotion"]])

In [ ]:
embeddings = np.array(embeddings)

In [ ]:
pca = PCA(n_components=2)

embeddings_2d = pca.fit_transform(
    embeddings
)

In [ ]:
print(
    pca.explained_variance_ratio_
)

In [ ]:
unique = sorted(set(labels))

for label in unique:
    idx = np.array(labels)==label
    plt.scatter(
        embeddings_2d[idx,0],
        embeddings_2d[idx,1],
        label=id2label[label]
    )

plt.legend()
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Whisper Embeddings - Original Model")
plt.show()

In [ ]:
embeddings = []
labels = []

In [ ]:
for sample in test_dataset:

    input_values = feature_extractor(sample["audio"]["array"], sampling_rate=sample["audio"]["sampling_rate"], return_tensors="pt").input_features
    input_values = input_values.to(model_ft.device)

    with torch.no_grad():
        outputs = model_ft(
            input_features=input_values,
            output_hidden_states=True
        )

    hidden = outputs.hidden_states[-1]
    embedding = hidden.mean(dim=1)
    embeddings.append(embedding.squeeze().cpu().numpy())
    labels.append(label2id[sample["emotion"]])

In [ ]:
embeddings = np.array(embeddings)

In [ ]:
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

In [ ]:
print(pca.explained_variance_ratio_)

In [ ]:
unique = sorted(set(labels))

for label in unique:
    idx = np.array(labels)==label
    plt.scatter(
        embeddings_2d[idx,0],
        embeddings_2d[idx,1],
        label=id2label[label]
    )

plt.legend()
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Whisper Embeddings - Finetuned Model")
plt.show()